# 03 - Neural Network Model

Prediction of Daily Mean TEC using Solar and Geomagnetic Indices

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
ROOT = Path.cwd().parent

DATA = ROOT / 'data' / 'processed' / 'master_dataset.csv'
RESULTS = ROOT / 'results'
MODELS = ROOT / 'models'

RESULTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df['date'] = pd.to_datetime(df['date'])

df.head()

## Feature Selection

In [ ]:
features = [
    'ssn',
    'kp_mean',
    'ap_daily',
    'f107_obs',
    'dst_daily_mean'
]

target = 'daily_mean_tec'

data = df[features + [target]].dropna()

X = data[features]
y = data[target]

## Train / Validation / Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print('Training:', len(X_train))
print('Validation:', len(X_val))
print('Testing:', len(X_test))

## Standardization

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Neural Network Architecture

In [ ]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

## Model Training

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

## Learning Curve

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Model Evaluation

In [ ]:
predictions = model.predict(X_test).flatten()

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'MAE  : {mae:.3f}')
print(f'RMSE : {rmse:.3f}')
print(f'R²   : {r2:.3f}')

## Observed vs Predicted

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test, predictions, alpha=0.7)
mn = min(y_test.min(), predictions.min())
mx = max(y_test.max(), predictions.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('Observed TEC')
plt.ylabel('Predicted TEC')
plt.grid(True)
plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
residuals = y_test - predictions

plt.figure(figsize=(8,5))
plt.scatter(predictions, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted TEC')
plt.ylabel('Residual')
plt.grid(True)
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
results = pd.DataFrame({
    'Observed': y_test.values,
    'Predicted': predictions,
    'Residual': residuals
})

results.to_csv(RESULTS / 'nn_predictions.csv', index=False)

model.save(MODELS / 'tec_neural_network.keras')

print('Saved predictions and trained model.')